In [0]:
%run ./config

In [0]:
import requests

TIINGO_KEY = dbutils.secrets.get(SECRET_SCOPE, "tiingo_api_key")

resp = requests.get(
    "https://api.tiingo.com/tiingo/daily/AAPL/prices",
    params={"startDate": "2024-01-02", "endDate": "2024-01-05"},
    headers={"Authorization": f"Token {TIINGO_KEY}"},   # key goes in a header, not the URL
    timeout=30,
)
print("HTTP status:", resp.status_code)
resp.json()[:2]

In [0]:
import json, os, time

START, END = "2024-01-01", "2025-12-31"
SNAPSHOT = "2026-11-15"                                   # one frozen download, never re-pulled
LANDING = f"/Volumes/{CATALOG}/bronze/raw/tiingo_prices/snapshot={SNAPSHOT}"
os.makedirs(LANDING, exist_ok=True)

tickers = [r.ticker for r in spark.table(f"{CATALOG}.gold.ref_universe").select("ticker").collect()]

downloaded, skipped = [], []
for ticker in sorted(tickers):
    path = f"{LANDING}/{ticker}.json"
    if os.path.exists(path):                              # already landed in an earlier run
        skipped.append(ticker); continue

    resp = requests.get(f"https://api.tiingo.com/tiingo/daily/{ticker}/prices",
                        params={"startDate": START, "endDate": END},
                        headers={"Authorization": f"Token {TIINGO_KEY}"}, timeout=60)
    if resp.status_code == 429:                            # hourly limit reached
        print(f"⏸️  Rate limit reached at {ticker}. Re-run this cell in about an hour.")
        break
    resp.raise_for_status()                                # any other error: stop and show it

    with open(path, "w") as f:
        f.write(resp.text)                                 # exactly as received
    downloaded.append(ticker)
    time.sleep(1)                                          # be polite to the API

landed = len(os.listdir(LANDING))
print(f"This run: downloaded {len(downloaded)}, skipped {len(skipped)}")
print(f"Landed so far: {landed} of {len(tickers)} tickers")

In [0]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

RUN_ID = str(uuid.uuid4())
EXPECTED_DAYS = 502            # ~252 trading days in 2024 + ~250 in 2025

# 0) Every ticker must have landed before bronze is built
files = sorted(f for f in os.listdir(LANDING) if f.endswith(".json"))
assert len(files) == len(tickers), f"Only {len(files)} of {len(tickers)} tickers landed. Re-run Cell 3 first."

# Expected side of the gate: count records in each file directly (no Spark)
file_counts = {f[:-5]: len(json.load(open(f"{LANDING}/{f}"))) for f in files}

# 1) Read every file in parallel, with an explicit schema (no guessing)
schema = """date STRING, open DOUBLE, high DOUBLE, low DOUBLE, close DOUBLE, volume BIGINT,
            adjOpen DOUBLE, adjHigh DOUBLE, adjLow DOUBLE, adjClose DOUBLE, adjVolume BIGINT,
            divCash DOUBLE, splitFactor DOUBLE"""
raw = (spark.read.schema(schema).option("multiLine", True).json(LANDING)
       .withColumn("_source_file", F.col("_metadata.file_path"))
       .withColumn("ticker", F.regexp_extract("_source_file", r"/([^/]+)\.json$", 1)))

# 2) Lineage columns, then append to bronze
(raw.withColumn("_source_system", F.lit("tiingo"))
    .withColumn("_snapshot", F.lit(SNAPSHOT))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_run_id", F.lit(RUN_ID))
    .write.mode("append").option("mergeSchema", "true")
    .saveAsTable(f"{CATALOG}.bronze.tiingo_prices"))

# 3) Gate: files → bronze
landed = (spark.table(f"{CATALOG}.bronze.tiingo_prices").filter(F.col("_run_id") == RUN_ID)
          .groupBy("ticker").count().collect())
bronze_counts = {r.ticker: r["count"] for r in landed}

def row(check, sev, exp, act, ok):
    return (RUN_ID, "files_to_bronze", "bronze.tiingo_prices", check, sev, str(exp), str(act),
            ok, datetime.now(timezone.utc))

short = sorted(t for t, n in bronze_counts.items() if n < EXPECTED_DAYS - 5)
results = [
    row("every_file_loaded", "error", len(file_counts), len(bronze_counts),
        "PASS" if set(file_counts) == set(bronze_counts) else "FAIL"),
    row("row_count_matches_files", "error", sum(file_counts.values()), sum(bronze_counts.values()),
        "PASS" if file_counts == bronze_counts else "FAIL"),
    row("full_history_per_ticker", "warn", f">= {EXPECTED_DAYS - 5} days each", short or "all complete",
        "PASS" if not short else "WARN"),
]
gate = spark.createDataFrame(results, spark.table(f"{CATALOG}.audit.gate_results").schema)
gate.write.mode("append").saveAsTable(f"{CATALOG}.audit.gate_results")
display(gate)
assert all(r[7] != "FAIL" for r in results), "❌ Gate files_to_bronze FAILED"
print(f"✅ bronze.tiingo_prices: {sum(bronze_counts.values()):,} rows for {len(bronze_counts)} tickers")